# Train CEQ attention from scratch, then push to the Hugging Face Hub

The attention here is a signed path sum and nothing else -- no softmax on the
output path:

```
out = v + A v + A^2 v                A strictly causal, SIGNED
A   = rho * (softmax(w) - lam * softmax(-w)) / (1 + lam)      w = q k^T / sqrt(d)
```

**The default operator is `sgate` at rho=1.5, lam=0.10, hops=2** -- the point that
measured val loss 1.0334x softmax over 5 seeds at 3,319,296 parameters on both
arms. The older `A = rho * w / sum_j |w_ij|` form is still selectable as
`operator='signed'` and measured **1.337x** with the gap widening as the budget
grew; it is the negative control, not the default. Setting `rho` or `hops`
without naming `operator` is refused, because a `config.json` written before the
`operator` key existed looks exactly like that and would load as the wrong
operator.

Signedness at `lam = 0.10` is **emergent, not structural**: at `nn.Linear`
initialization the negative fraction is about 4e-04 and it grows with the
within-row logit spread (5.25e-02 at unit-variance logits). `lam = 1.0` makes it
structural -- rows sum to exactly zero -- at the cost of annihilating the
constant vector. `modeling_ceq.COSTS['signedness']` carries the numbers.

Before allocating a runtime, run the package smoke test locally -- it is CPU
only, needs no network, and takes seconds:

```
python -m ceq.hf.smoke
```

## Read this before you allocate a runtime

**The Triton kernel is NOT on this path.** `ceq/mz_kernel.py` is forward-only --
it has no backward pass -- so it cannot appear in a training step. Training is
pure PyTorch. That is why a free-tier **T4 (sm_75) can run this notebook** even
though the kernel needs compute capability 8.0+ and ships Linux-only wheels.

**Memory grows as O(S^2), not O(S).** SDPA never forms the [S,S] score matrix;
this operator materializes it and autograd retains ~3.9 such tensors per layer.
Measured forward+backward peak, RTX 4060 Laptop (sm_89), B=4 d=256 L=4 H=4, fp32:

| seq | softmax | signed | ratio |
|---:|---:|---:|---:|
| 128 | 36.1 MiB | 51.9 MiB | 1.44x |
| 256 | 74.1 MiB | 128.7 MiB | 1.74x |
| 512 | 143.5 MiB | 379.9 MiB | 2.65x |
| 1024 | 285.8 MiB | 1271.2 MiB | 4.45x |
| 2048 | 570.3 MiB | 4596.0 MiB | 8.06x |

bf16 autocast does NOT halve this: measured signed-arm bf16/fp32 peak is
0.748 / 0.772 / 0.797 at seq 256 / 512 / 1024, never near 0.5.

The 1.31x-1.94x figure on record is the forward-only kernel in the 128-512 band.
It does not describe training at 2048.

**At 0.5B and seq 2048 this fits NO Colab GPU at batch 1** without gradient
checkpointing: 36.31 GiB needed against an A100-40GB's 33.53 GiB budget, and
230.01 GiB at batch 8. With `GRAD_CHECKPOINT = True` it reaches batch 15 on an
A100-40GB and batch 7 on an L4-24GB. The softmax control needs none of this.

**Parity with softmax is not established.** At 3.3M parameters, matched
everything, 800 steps on TinyStories: signed val loss 2.0064 against softmax
1.4282 -- 1.405x. Run the softmax control in this notebook before you believe
any number this produces.


In [ ]:
# 1. Install. Colab already has torch; this pins the rest.
!pip -q install -U 'transformers>=5.0' datasets huggingface_hub accelerate

import torch, platform
print('torch', torch.__version__, platform.system(), platform.machine())


In [ ]:
# 2. Get the code. Set REPO_URL to your clone of this project.
#    Everything the notebook calls lives in ceq/hf/train.py, which is tested
#    on cpu and cuda in tests/chase/test_colab_chain.py -- the notebook itself
#    holds no logic worth trusting on its own.
import os, sys

REPO_URL = ''      # e.g. 'https://github.com/<you>/<repo>.git'
REPO_DIR = 'ceq-project'

if REPO_URL and not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
if os.path.isdir(REPO_DIR):
    sys.path.insert(0, os.path.abspath(REPO_DIR))

from ceq.hf import train as ceq_train
from ceq import sizing
print('ceq loaded from', os.path.dirname(ceq_train.__file__))


In [ ]:
# 3. Which accelerator did Colab give us, and does the run fit on it?
#    T4 sm_75 16GB (free) | L4 sm_89 ~22.5GiB | A100 sm_80 40GB
GPU = 'T4-16GB'
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cc = torch.cuda.get_device_capability(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    for key in sizing.GPUS:
        if key.split('-')[0].lower() in name.lower():
            GPU = key
    print(f'{name}  sm_{cc[0]}{cc[1]}  {total:.1f} GiB  -> budgeting as {GPU}')
else:
    print('NO GPU. Runtime > Change runtime type > GPU. CPU will work and will be slow.')


In [ ]:
# 4. THE PREFLIGHT. Refuses a shape that cannot fit, before anything downloads.
CONFIG = dict(ceq_train.DEFAULTS)      # hidden 512, 8 layers, 8 heads, seq 512, batch 8
GRAD_CHECKPOINT = False

# For the 0.5B target, uncomment these. seq 2048 batch 8 needs 230.01 GiB of
# activations without checkpointing and 21.61 GiB with it -- the preflight will
# print both numbers and stop the run.
# CONFIG = dict(hidden_size=1280, n_layers=24, n_heads=20, seq=2048, batch=8,
#               vocab_size=256)
# GRAD_CHECKPOINT = True

ok, msg = ceq_train.preflight(gpu=GPU, grad_checkpoint=GRAD_CHECKPOINT, **CONFIG)
print(msg)
assert ok, 'This shape does not fit. Lower seq or batch, or set GRAD_CHECKPOINT=True.'


In [ ]:
# 5. Open data. TinyStories, Dolma, The Pile, FineWeb-Edu and wikitext all
#    resolved without a token when probed; TinyStories is the smallest.
#    Streamed, not downloaded: Colab's disk is shared with the checkpoints.
DATA = 'corpus.txt'
if not os.path.exists(DATA):
    text = ceq_train.load_open_text('roneneldan/TinyStories', max_bytes=32*1024*1024)
    open(DATA, 'w', encoding='utf-8').write(text)
print(os.path.getsize(DATA) / 1024**2, 'MiB of open text')


In [ ]:
# 6. Train. `row_l1_min` in the log means DIFFERENT things per operator, so read
#    it against the operator you selected.
#      operator='signed'  -- a BLOW-UP ALARM. The backward of A = rho*w/sum|w|
#        carries a factor 1/l1, measured at exactly 10x per decade over six
#        decades; over 800 real steps the minimum reached 9.35e-07 while the
#        GLOBAL gradient norm stayed at 1.21, so the norm cannot see it.
#      operator='sgate' (the default) -- a LOGIT-SPREAD diagnostic. sgate divides
#        by the constant 1 + lam and by nothing data-dependent, and the raw-logit
#        row L1 has a measured constructive floor of 1.227272 at every d and S.
#        There is no 1/l1 to blow up. What this number tracks instead is whether
#        the logits are spread enough for the operator to have negative entries
#        at all -- see modeling_ceq.COSTS['signedness'].
OUT = 'ceq-run'
record = ceq_train.train(out_dir=OUT, steps=2000, device='cuda' if torch.cuda.is_available() else 'cpu',
                         data_path=DATA, grad_checkpoint=GRAD_CHECKPOINT,
                         gpu=GPU if torch.cuda.is_available() else None,
                         log_every=50, **CONFIG)
print('params', f"{record['n_params']:,}", ' peak', record['peak_bytes'] / 1024**3, 'GiB')
print('final loss', record['losses'][-1], ' worst row L1', min(record['row_l1_min']))


In [ ]:
# 7. Verify it loads the way a downloader loads it -- through the Auto class,
#    with trust_remote_code, on CPU. configuration_ceq.py and modeling_ceq.py
#    are in the output directory because register_for_auto_class was called;
#    without them auto_map points at nothing and the repo raises on load.
from transformers import AutoModelForCausalLM
print(sorted(os.listdir(OUT)))
assert {'configuration_ceq.py', 'modeling_ceq.py'} <= set(os.listdir(OUT))

m = AutoModelForCausalLM.from_pretrained(OUT, trust_remote_code=True).to('cpu').eval()
assert not m.lm_head.weight.is_meta, 'lm_head is on the meta device'
print(m.generate(torch.randint(0, 256, (1, 8)), max_new_tokens=16, do_sample=False))


In [ ]:
# 8. Push. The token is read from the login prompt or the HF_TOKEN environment
#    variable and is never written into this notebook.
from huggingface_hub import login
login()

REPO_ID = ''       # e.g. '<your-username>/ceq-attention-0.5b'
assert REPO_ID, 'set REPO_ID'
print(ceq_train.push(OUT, REPO_ID, private=True))


## What this checkpoint cannot do, stated so nobody discovers it later

1. **No KV-cache decoding.** `(A^h v)_i = sum_{j<i} A_ij (A^{h-1} v)_j` needs the
   whole prefix's hop vectors; during decode the operator row is `[1, N]` and
   `A^2` is undefined. `use_cache` is forced off and `generate()` recomputes the
   full prefix each step -- exact, and O(N^2) per token. `ceq/hopcache.py` fixes
   this at K cached vectors per position and is not wired in here.

2. **No Serverless Inference, no model-page widget.** Inference Providers do not
   run remote-code models. Every downloader must pass `trust_remote_code=True`,
   which is an arbitrary-code-execution opt-in that some organisations block.

3. **Untied embeddings, deliberately.** `PreTrainedModel.is_remote_code()` is
   `cls._auto_class is not None`, so calling `register_for_auto_class()` -- the
   only way `save_pretrained` copies these .py files -- puts the model on a load
   path that refuses to tie and leaves `lm_head.weight` on the META device with
   no exception raised. Untying costs one extra vocab x d matrix and removes the
   failure entirely.

4. **Parity with softmax is unproven.** Run the control. `ceq/lm.py` has both arms
   at identical parameter counts and `ceq.lm.compare()` runs them head to head.
